# Clase 14C — Mini ChatGPT con RAG usando el dataset real del curso
**Autor:** Josef Rodríguez

Esta clase integra todo lo aprendido:

- chunking
- embeddings
- vector search
- construcción de contexto
- respuesta con modelo generativo

Dataset utilizado:

`https://raw.githubusercontent.com/josefrodrim/ML-course/main/data/planes_gobierno_texto.csv`

# 1. Objetivo de la clase

Construir un sistema estilo **mini ChatGPT con RAG** sobre planes de gobierno.

## Flujo final

```text
Dataset real
   ↓
Chunking
   ↓
Embeddings
   ↓
Índice vectorial
   ↓
Consulta del usuario
   ↓
Top-K chunks
   ↓
Prompt
   ↓
LLM
   ↓
Respuesta
```

# 2. Cargar datos y preparar chunks

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/josefrodrim/ML-course/main/data/planes_gobierno_texto.csv"
df = pd.read_csv(url)
df = df.rename(columns={df.columns[0]: "presidente", df.columns[1]: "texto_plan"})
df["texto_plan"] = df["texto_plan"].fillna("").astype(str)

df.head()

,presidente,texto_plan
0,PABLO ALFONSO LOPEZ CHAU NAVA,formato del resumen del plan de gobierno el fo...
1,RONALD DARWIN ATENCIO SOTOMAYOR,formato del resumen del plan de gobierno el fo...
2,CESAR ACUÑA PERALTA,formato del resumen del plan de gobierno el fo...
3,JOSE DANIEL WILLIAMS ZAPATA,formato del resumen del plan de gobierno el fo...
4,ALVARO GONZALO PAZ DE LA BARRA FREIGEIRO,formato del resumen del plan de gobierno el fo...


In [2]:
def chunk_text(text, chunk_size=180, overlap=40):
    words = text.split()
    chunks = []
    step = max(chunk_size - overlap, 1)
    for i in range(0, len(words), step):
        part = words[i:i+chunk_size]
        if not part:
            continue
        chunks.append(" ".join(part))
        if i + chunk_size >= len(words):
            break
    return chunks

rows = []
for _, row in df.iterrows():
    chunks = chunk_text(row["texto_plan"], chunk_size=180, overlap=40)
    for j, ch in enumerate(chunks):
        rows.append({
            "presidente": row["presidente"],
            "chunk_id": j,
            "chunk_text": ch
        })

chunks_df = pd.DataFrame(rows)
chunks_df.head()

,presidente,chunk_id,chunk_text
0,PABLO ALFONSO LOPEZ CHAU NAVA,0,formato del resumen del plan de gobierno el fo...
1,PABLO ALFONSO LOPEZ CHAU NAVA,1,de dichos objetivos. problemas objetivos indic...
2,PABLO ALFONSO LOPEZ CHAU NAVA,2,reciba la atención que necesita en el momento ...
3,PABLO ALFONSO LOPEZ CHAU NAVA,3,"salud y/o remodelados y oportunos, puestos en ..."
4,PABLO ALFONSO LOPEZ CHAU NAVA,4,de afectan la salud de vacunación en menores s...


# 3. Embeddings e índice vectorial

In [3]:
# Si estás en un entorno limpio, descomenta:
# !pip install -q sentence-transformers faiss-cpu transformers accelerate sentencepiece

In [4]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_embeddings = embed_model.encode(
    chunks_df["chunk_text"].tolist(),
    show_progress_bar=True
)
chunk_embeddings = np.array(chunk_embeddings).astype("float32")

dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(chunk_embeddings)

index.ntotal

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/28 [00:00<?, ?it/s]

881

# 4. Retriever

In [5]:
def retrieve_chunks(query, top_k=5):
    q_emb = embed_model.encode([query]).astype("float32")
    distances, indices = index.search(q_emb, top_k)
    out = chunks_df.iloc[indices[0]].copy()
    out["distance_l2"] = distances[0]
    return out[["presidente", "chunk_id", "chunk_text", "distance_l2"]]

retrieve_chunks("propuestas sobre educación pública", top_k=5)

,presidente,chunk_id,chunk_text,distance_l2
515,VLADIMIR ROY CERRON ROJAS,23,de 16. • incrementar desigual y limitado unive...,0.747294
532,VLADIMIR ROY CERRON ROJAS,40,"programas de territorios, lo que públicas arti...",0.761905
572,VLADIMIR ROY CERRON ROJAS,80,de 11. • incorporar educación y cultura ciudad...,0.786557
675,MARIA SOLEDAD PEREZ TELLO DE RODRIGUEZ,24,"asociación (social, % de jóvenes 15-29 al comu...",0.790544
200,ROBERTO HELBERT SANCHEZ PALOMINO,13,con programa nacional ¿ # de jóvenes públicas....,0.797957


# 5. Prompt builder

En un sistema RAG serio, el prompt debe:

1. delimitar el contexto,
2. restringir la respuesta a la evidencia,
3. dejar explícito qué hacer si no hay soporte.

In [6]:
def build_prompt(query, retrieved_df):
    context = "\n\n".join(
        [f"[{r.presidente} | chunk {r.chunk_id}] {r.chunk_text}" for r in retrieved_df.itertuples()]
    )
    prompt = f"""Eres un asistente analítico que responde preguntas sobre planes de gobierno presidenciales.
Usa únicamente el contexto proporcionado.
Si la respuesta no se encuentra claramente en el contexto, responde: "No lo sé con la evidencia recuperada".

Contexto:
{context}

Pregunta:
{query}

Respuesta:
"""
    return prompt

In [7]:
prompt_preview = build_prompt(
    "¿Qué candidatos hablan de salud pública?",
    retrieve_chunks("salud pública, hospitales y atención primaria", top_k=4)
)

print(prompt_preview[:2500])

Eres un asistente analítico que responde preguntas sobre planes de gobierno presidenciales.
Usa únicamente el contexto proporcionado.
Si la respuesta no se encuentra claramente en el contexto, responde: "No lo sé con la evidencia recuperada".

Contexto:
[VLADIMIR ROY CERRON ROJAS | chunk 8] comunitario, familias y resueltas en el primer equipo de médicos generando altos comunidades, y nivel de atención de la familia a niveles de acción preventiva • tasa de cada territorio enfermedades sobre los hospitalizaciones por definido, evitables, determinantes enfermedades garantizando saturación sociales de la prevenibles cobertura hospitalaria y salud, fortaleciendo • prevalencia de progresiva de la mayores costos el primer nivel de anemia y desnutrición población. sociales y atención como eje infantil • lograr que la económicos para el del sistema público • número de visitas mayoría de estado y las de salud. domiciliarias atenciones de familias. preventivas realizadas salud se resuelvan en el

# 6. Integración con un modelo generativo

Para docencia y pruebas locales, puedes usar modelos pequeños de Hugging Face.  
Dependiendo del entorno, algunos modelos pueden requerir más memoria.

## Opción docente liviana
- `google/flan-t5-base`

## Otras opciones
- modelos instruccionales pequeños
- APIs externas si el curso lo permite

In [9]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'ExaoneMoeForCausalLM', 'FalconForCausalLM', 'FalconH1ForCausalL

# 7. Función final del sistema RAG

In [10]:
def rag_answer(query, top_k=5, max_new_tokens=220):
    retrieved = retrieve_chunks(query, top_k=top_k)
    prompt = build_prompt(query, retrieved)
    response = generator(prompt, max_new_tokens=max_new_tokens)[0]["generated_text"]
    return {
        "query": query,
        "retrieved": retrieved,
        "prompt": prompt,
        "answer": response
    }

In [11]:
result = rag_answer("¿Qué candidatos mencionan educación pública y mejora de la calidad educativa?", top_k=4)
print(result["answer"])

Token indices sequence length is longer than the specified maximum sequence length for this model (2033 > 512). Running this sequence through the model will result in indexing errors
Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Eres un asistente analítico que responde preguntas sobre planes de gobierno presidenciales.
Usa únicamente el contexto proporcionado.
Si la respuesta no se encuentra claramente en el contexto, responde: "No lo sé con la evidencia recuperada".

Contexto:
[PABLO ALFONSO LOPEZ CHAU NAVA | chunk 7] abandono y baja educación de implemente estrategias conclusión de la calidad a lo largo orientadas a que la educación básica de la vida, con totalidad de los enfoque territorial, estudiantes de que desarrolle el educación primaria máximo potencial comprendan lo que de los estudiantes, lean asegure su tránsito a la educación superior y al trabajo, cierre brechas de exclusión y bajo rendimiento, y fortalezca a las instituciones educativas como espacios de transformación e innovación, con participación de estudiantes y familias 15. baja calidad, 15. garantizar una 15. garantizar que el 15. 1 acceso limitado, educación de sistema educativo abandono y baja calidad a lo largo implemente estrategias co

In [12]:
result["retrieved"][["presidente", "chunk_id", "distance_l2"]]

,presidente,chunk_id,distance_l2
7,PABLO ALFONSO LOPEZ CHAU NAVA,7,0.590459
193,ROBERTO HELBERT SANCHEZ PALOMINO,6,0.603419
6,PABLO ALFONSO LOPEZ CHAU NAVA,6,0.643155
201,ROBERTO HELBERT SANCHEZ PALOMINO,14,0.648651


# 8. Casos de uso sobre el dataset real

Prueba preguntas como:

- ¿Qué candidatos hablan de salud pública?
- ¿Qué propuestas aparecen sobre telemedicina?
- ¿Qué candidatos mencionan minería y crecimiento económico?
- ¿Qué candidatos proponen mejoras en educación básica?
- ¿Qué evidencias concretas recupera el sistema?

Una buena práctica docente es comparar:
1. la respuesta final,
2. los chunks recuperados,
3. la trazabilidad hacia la evidencia.

In [13]:
preguntas_demo = [
    "¿Qué candidatos hablan de hospitales y salud pública?",
    "¿Qué candidatos mencionan telemedicina?",
    "¿Qué candidatos hablan de minería y exportaciones?",
    "¿Qué candidatos proponen mejoras en educación básica?"
]

for p in preguntas_demo:
    print("="*100)
    print("PREGUNTA:", p)
    out = rag_answer(p, top_k=4, max_new_tokens=180)
    print("RESPUESTA:", out["answer"])
    print()

PREGUNTA: ¿Qué candidatos hablan de hospitales y salud pública?


Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


RESPUESTA: Eres un asistente analítico que responde preguntas sobre planes de gobierno presidenciales.
Usa únicamente el contexto proporcionado.
Si la respuesta no se encuentra claramente en el contexto, responde: "No lo sé con la evidencia recuperada".

Contexto:
[VLADIMIR ROY CERRON ROJAS | chunk 8] comunitario, familias y resueltas en el primer equipo de médicos generando altos comunidades, y nivel de atención de la familia a niveles de acción preventiva • tasa de cada territorio enfermedades sobre los hospitalizaciones por definido, evitables, determinantes enfermedades garantizando saturación sociales de la prevenibles cobertura hospitalaria y salud, fortaleciendo • prevalencia de progresiva de la mayores costos el primer nivel de anemia y desnutrición población. sociales y atención como eje infantil • lograr que la económicos para el del sistema público • número de visitas mayoría de estado y las de salud. domiciliarias atenciones de familias. preventivas realizadas salud se resu

Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


RESPUESTA: Eres un asistente analítico que responde preguntas sobre planes de gobierno presidenciales.
Usa únicamente el contexto proporcionado.
Si la respuesta no se encuentra claramente en el contexto, responde: "No lo sé con la evidencia recuperada".

Contexto:
[MESIAS ANTONIO GUEVARA AMASIFUEN | chunk 2] 30 minutos. 4. centralización de 4. establecer cinco 4. número de 4. 5 la atención de casos macrorregiones con macrorregiones con médicos complejos hospitales de nivel hospitales de nivel únicamente en iv especializados. iv operativos. lima. 5. zonas rurales sin 5. implementar una 5. número de 5. 2,500 especialistas, red de telemedicina establecimientos atendidas solo por con 2,500 rurales con servicio médicos generales. establecimientos de telemedicina. rurales. 6. brecha digital en 6. desarrollar una 6. porcentaje de 6. 100% la educación, con un plataforma escuelas con 40% de escuelas sin educativa nacional conectividad internet funcional. con contenidos de funcional. ciencia, te

Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


RESPUESTA: Eres un asistente analítico que responde preguntas sobre planes de gobierno presidenciales.
Usa únicamente el contexto proporcionado.
Si la respuesta no se encuentra claramente en el contexto, responde: "No lo sé con la evidencia recuperada".

Contexto:
[ANTONIO ORTIZ VILLANO | chunk 14] según una 7. fomentar el 7. % de padres y madres de 7. reducir al 40 encuesta realizada involucramiento de familia que participan % la proporción por la los padres en activamente en al menos de padres con organización la educación de sus una actividad bajo o nulo "ciudadanos al día" hijos, como la educativa escolar anual involucramiento, en el 2019, el creación de logrando que al 67% de los padres programas de menos el 60 % en el perú se capacitación para de los padres involucran poco padres y la participe o nada en la implementación de activamente en la educación de sus plataformas educación de sus hijos (garcía, virtuales para hijos. 2019). esto se debe mejorar la en parte a la falta de co

# 9. Evaluación cualitativa del sistema

Al evaluar un mini chatbot con RAG no basta con leer la respuesta final.  
Debemos revisar:

1. **Retrieval**: ¿recuperó evidencia pertinente?
2. **Grounding**: ¿la respuesta está soportada por el contexto?
3. **Cobertura**: ¿faltó información importante?
4. **Faithfulness**: ¿inventó algo no presente?
5. **Claridad**: ¿respondió de forma útil?

## Posibles mejoras
- reranking
- chunking semántico por párrafos
- filtros por candidato
- ranking híbrido TF-IDF + embeddings
- modelos generativos más fuertes

# 10. Cierre de la masterclass

Con esta clase cerramos el bloque moderno de NLP aplicado:

- En la clase 14A representamos texto con embeddings.
- En la clase 14B construimos retrieval semántico con chunks y FAISS.
- En la clase 14C integramos todo en un mini sistema conversacional con RAG.

## Frase final
**El verdadero valor de un sistema de IA aplicada no está solo en generar texto, sino en generar respuestas sustentadas por evidencia recuperada.**